# Proyek Klasifikasi Gambar: Intel Image Classification
- **Nama:** Muhammad Ragil
- **Email:** mhmmdragilpy
- **ID Dicoding:** mhmmdragilpy

## Import Semua Packages/Library yang Digunakan

In [ ]:
!nvidia-smi 2>/dev/null || echo 'GPU belum aktif. Aktifkan via Runtime > Change runtime type > T4 GPU'
!pip install -q split-folders pipreqs
!pip install -q tensorflowjs --no-deps

In [ ]:
import os
import shutil
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
import splitfolders

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Data Preparation

### Data Loading

In [ ]:
# Download dan ekstrak dataset dari Kaggle
dataset_zip = 'intel-image-classification.zip'
dataset_dir = 'dataset'

if not os.path.exists(dataset_dir):
    if not os.path.exists(dataset_zip):
        !curl -L -o intel-image-classification.zip https://www.kaggle.com/api/v1/datasets/download/puneet6060/intel-image-classification
    with zipfile.ZipFile(dataset_zip, 'r') as z:
        z.extractall(dataset_dir)
    print('Dataset berhasil diekstrak.')
else:
    print('Dataset sudah tersedia.')

In [ ]:
# Gabungkan seg_train dan seg_test ke satu direktori untuk di-split ulang
raw_dir = 'raw_combined'
classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

for cls in classes:
    os.makedirs(os.path.join(raw_dir, cls), exist_ok=True)

for src in [os.path.join(dataset_dir, 'seg_train', 'seg_train'),
            os.path.join(dataset_dir, 'seg_test', 'seg_test')]:
    if os.path.exists(src):
        for cls in classes:
            s = os.path.join(src, cls)
            d = os.path.join(raw_dir, cls)
            if os.path.exists(s):
                for f in os.listdir(s):
                    if not os.path.exists(os.path.join(d, f)):
                        shutil.copy2(os.path.join(s, f), os.path.join(d, f))

# Verifikasi jumlah gambar dan variasi resolusi asli
resolutions = set()
total = 0
for cls in classes:
    files = [f for f in os.listdir(os.path.join(raw_dir, cls))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    total += len(files)
    print(f'{cls}: {len(files)} gambar')
    for fn in files:
        with Image.open(os.path.join(raw_dir, cls, fn)) as img:
            resolutions.add(img.size)

print(f'\nTotal gambar: {total}')
print(f'Variasi resolusi asli: {len(resolutions)} dimensi unik')
for r in sorted(resolutions)[:8]:
    print(f'  {r[0]}x{r[1]}')

In [ ]:
# Tampilkan sampel gambar dari setiap kelas
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, cls in enumerate(classes):
    folder = os.path.join(raw_dir, cls)
    img_path = os.path.join(folder, os.listdir(folder)[0])
    img = Image.open(img_path)
    ax = axes[i // 3][i % 3]
    ax.imshow(img)
    ax.set_title(f'{cls} ({img.size[0]}x{img.size[1]})', fontweight='bold')
    ax.axis('off')
plt.suptitle('Sampel Gambar Tiap Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Data Preprocessing

#### Split Dataset

In [ ]:
# Split dataset: 80% train, 10% val, 10% test
split_dir = 'split_dataset'

if not os.path.exists(split_dir) or len(os.listdir(split_dir)) < 3:
    splitfolders.ratio(raw_dir, output=split_dir, seed=42,
                       ratio=(0.80, 0.10, 0.10))

train_dir = os.path.join(split_dir, 'train')
val_dir   = os.path.join(split_dir, 'val')
test_dir  = os.path.join(split_dir, 'test')

print(f'Train: {train_dir}')
print(f'Val  : {val_dir}')
print(f'Test : {test_dir}')

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

# Augmentasi hanya pada data training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Normalisasi saja untuk validasi dan test
val_datagen  = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True)

val_gen = val_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

test_gen = test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

NUM_CLASSES = train_gen.num_classes
class_names = list(train_gen.class_indices.keys())
print(f'Jumlah kelas: {NUM_CLASSES}')
print(f'Kelas: {class_names}')

## Modelling

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(150,150,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

In [ ]:
# Custom callback: hentikan training jika akurasi sudah tercapai
class AccuracyThreshold(Callback):
    def __init__(self, threshold=0.96):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy', 0) >= self.threshold and logs.get('val_accuracy', 0) >= self.threshold:
            print(f'\nAkurasi target tercapai di epoch {epoch+1}. Training dihentikan.')
            self.model.stop_training = True

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, mode='max',
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3,
                      min_lr=1e-6, verbose=1),
    AccuracyThreshold(threshold=0.96)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

## Evaluasi dan Visualisasi

In [ ]:
# Evaluasi pada test set
test_loss, test_acc = model.evaluate(test_gen)
print(f'\nTest Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

In [ ]:
# Plot akurasi dan loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc)+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc, 'b-o', label='Training')
ax1.plot(epochs, val_acc, 'g-s', label='Validation')
ax1.axhline(y=0.95, color='r', linestyle='--', alpha=0.6, label='Target 95%')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, loss, 'r-o', label='Training')
ax2.plot(epochs, val_loss, 'orange', marker='s', label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Konversi Model

In [ ]:
# Buat struktur folder submission
os.makedirs('submission/saved_model', exist_ok=True)
os.makedirs('submission/tflite', exist_ok=True)
os.makedirs('submission/tfjs_model', exist_ok=True)

# 1. SavedModel
try:
    model.export('submission/saved_model')
except Exception:
    model.save('submission/saved_model')
print('SavedModel berhasil disimpan.')

# 2. TF-Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('submission/tflite/model.tflite', 'wb') as f:
    f.write(tflite_model)

with open('submission/tflite/label.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')
print('TF-Lite model dan label.txt berhasil disimpan.')

# 3. TFJS
try:
    import tensorflowjs as tfjs
    tfjs.converters.save_keras_model(model, 'submission/tfjs_model')
    print('TFJS model berhasil disimpan.')
except Exception:
    model.save('temp_model.h5')
    !tensorflowjs_converter --input_format=keras temp_model.h5 submission/tfjs_model
    os.remove('temp_model.h5') if os.path.exists('temp_model.h5') else None
    print('TFJS model berhasil disimpan via CLI.')

## Inference

In [ ]:
# Inference menggunakan model TF-Lite
def tflite_predict(model_path, img_path, labels, size=(150,150)):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()

    img = Image.open(img_path).convert('RGB')
    img_resized = img.resize(size)
    arr = np.expand_dims(np.array(img_resized, dtype=np.float32)/255.0, axis=0)

    interpreter.set_tensor(inp[0]['index'], arr)
    interpreter.invoke()
    probs = interpreter.get_tensor(out[0]['index'])[0]
    idx = np.argmax(probs)
    return img, labels[idx], probs[idx]*100

# Ambil sampel dari tiap kelas di test set
samples = []
for cls in class_names:
    d = os.path.join(test_dir, cls)
    samples.append((cls, os.path.join(d, os.listdir(d)[0])))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (true_label, path) in enumerate(samples):
    img, pred, conf = tflite_predict('submission/tflite/model.tflite', path, class_names)
    ax = axes[i//3][i%3]
    ax.imshow(img)
    color = 'green' if true_label == pred else 'red'
    ax.set_title(f'Aktual: {true_label}\nPrediksi: {pred} ({conf:.1f}%)',
                 color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('Hasil Inference Model TF-Lite', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Generate requirements.txt
!pipreqs . --force --scan-notebooks --ignore dataset,raw_combined,split_dataset 2>/dev/null || true

if os.path.exists('requirements.txt'):
    shutil.copy2('requirements.txt', 'submission/requirements.txt')

readme = '''# Proyek Klasifikasi Gambar: Intel Image Classification
- Nama: Muhammad Ragil
- Email: mhmmdragilpy
- ID Dicoding: mhmmdragilpy
'''
with open('submission/README.md', 'w') as f:
    f.write(readme)

shutil.make_archive('submission', 'zip', 'submission')
print('submission.zip berhasil dibuat.')